# Goal 4 — Recovery Fine-Tuning with LoRA

**Objective:** Fine-tune the grafted Qwen2.5-0.5B models on Armenian text to recover
from embedding surgery and build Armenian language capability.

**Input models:** 3 grafted variants from Goal 3 (mean-init, heuristic-init, nearest-init)

**Training corpus:** hy_clean.txt (4.48 GB, 12.1M lines of cleaned CC-100 Armenian)

**Hardware:** Google Colab T4 (15.6 GB VRAM) or GCloud A100 (40 GB VRAM)

---

## What Recovery Fine-Tuning Does

After Goal 3's surgery, the model has:
- A tokenizer that efficiently encodes Armenian (1.69 tokens/word vs 7.81)
- New embeddings that are approximate initializations (not yet learned)
- Intact English knowledge in all other weights

Fine-tuning teaches the model to:
1. Map new Armenian token embeddings to meaningful representations
2. Route information through the transformer layers using Armenian tokens
3. Predict the next Armenian token from context (causal LM objective)

We compare all 3 init strategies to see which recovers fastest.

## 0. Setup

In [ ]:
# Install if needed (uncomment on fresh VM/Colab)
# !pip install transformers accelerate peft bitsandbytes datasets tqdm tabulate wandb

import torch
import os
import json
import time
import math
import random
import unicodedata
from pathlib import Path
from tqdm import tqdm

from transformers import (AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer, DataCollatorForLanguageModeling)
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

## 1. Configuration

```
FINE-TUNING APPROACH: LoRA vs Full Fine-Tuning
------------------------------------------------
Considered:
  (a) Full fine-tuning — update all 494M parameters
  (b) LoRA (Low-Rank Adaptation) — freeze base, train small adapter matrices
  (c) Full fine-tuning of embeddings only + LoRA on attention

Chose (c) — LoRA on attention + unfrozen embeddings — because:
  - The NEW embeddings (30,766 tokens) MUST be trained — they're just
    initializations. Freezing them would defeat the purpose of surgery.
  - LoRA on attention layers lets the model learn to route Armenian
    token information through the transformer without catastrophically
    forgetting English.
  - Full fine-tuning risks English catastrophic forgetting and requires
    more VRAM (full optimizer states for 494M params).
  - LoRA trains only ~2-5% of parameters, fitting comfortably on T4.
  - This combined approach is standard in tokenizer surgery literature
    (FOCUS, vocab extension papers).
```

```
LoRA CONFIGURATION: rank=16, alpha=32, target=q_proj,k_proj,v_proj,o_proj
-------------------------------------------------------------------------
Considered:
  rank: 4, 8, 16, 32, 64
  targets: attention only, attention+MLP, all linear

Chose rank=16, attention projections only because:
  - rank=16 is a good balance for a 0.5B model. Too low (4) limits
    capacity; too high (64) approaches full fine-tuning cost.
  - alpha=32 (2x rank) is the standard scaling that works well
    empirically across many LoRA papers.
  - Targeting q/k/v/o projections adapts the attention mechanism
    to process Armenian tokens, which is the core bottleneck.
  - Adding MLP layers (gate_proj, up_proj, down_proj) increases
    trainable params by 3x for diminishing returns at this scale.
  - We can always increase rank or add MLP targets if results
    are unsatisfactory.
```

```
TRAINING DATA: 500k lines from 12.1M (streaming)
--------------------------------------------------
Considered: 100k, 500k, 1M, full 12.1M

Chose 500k because:
  - Recovery fine-tuning doesn't need the full pretraining corpus.
    The model already has pretrained weights — we're teaching it
    to use new tokens, not learning language from scratch.
  - 500k lines × ~40 words × ~1.7 tokens/word ≈ 34M tokens.
    Literature suggests 10-50M tokens is sufficient for vocabulary
    recovery fine-tuning.
  - On T4: 500k lines ≈ 2-4 hours of training (manageable).
    Full 12.1M would take 20+ hours.
  - On A100: 500k lines ≈ 30-60 minutes.
  - 100k might underfit; 1M is viable if you have time.
```

```
LEARNING RATE: 2e-4 with cosine schedule
------------------------------------------
Considered: 1e-4, 2e-4, 5e-4, 1e-3

Chose 2e-4 because:
  - Standard LoRA learning rate (QLoRA paper uses 2e-4).
  - Higher than typical full fine-tuning (which uses ~2e-5)
    because LoRA adapters are initialized near zero and need
    larger steps to learn meaningful adaptations.
  - Cosine schedule with warmup avoids sharp initial updates
    that could destabilize the model.
  - The embedding layer gets its own LR (optionally higher)
    since new embeddings need more aggressive updates.
```

```
SEQUENCE LENGTH: 512 tokens
----------------------------
Considered: 256, 512, 1024, 2048

Chose 512 because:
  - With our grafted tokenizer (fertility 1.69), 512 tokens covers
    ~300 Armenian words — a substantial context window.
  - 1024+ doubles memory usage per sample for diminishing returns
    on recovery training (we're learning token representations,
    not long-range dependencies).
  - 256 might miss paragraph-level patterns.
  - 512 fits comfortably in T4 memory with LoRA + mixed precision.
```

Optional Colab setup note: if running this notebook in Google Colab, mount Google Drive manually before executing the training cells if your corpus is stored there.


In [ ]:
# CONFIGURATION
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "README.md").exists():
    REPO_ROOT = REPO_ROOT.parent

GOAL3_DIR = str(REPO_ROOT / "models" / "grafted_tokenizers")

MODEL_DIRS = {"heuristic_init": os.path.join(GOAL3_DIR, "heuristic_init"),
                   "mean_init": os.path.join(GOAL3_DIR, "mean_init"),
                "nearest_init": os.path.join(GOAL3_DIR, "nearest_init")}
TOKENIZER_DIR = os.path.join(GOAL3_DIR, "extended_tokenizer")

# Full cleaned corpus is intentionally not committed. Set HY_CLEAN_PATH
# to its location, or place it at data/hy_clean.txt.
CORPUS_PATH = os.environ.get("HY_CLEAN_PATH", str(REPO_ROOT / "data" / "hy_clean.txt"))
EVAL_PATH = str(REPO_ROOT / "data" / "sample" / "hy_sample_raw.txt")
OUTPUT_DIR = str(REPO_ROOT / "outputs" / "goal4_finetuned")

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# Training hyperparameters
MAX_TRAIN_LINES = 500_000
MAX_SEQ_LEN = 512
BATCH_SIZE = 4
GRAD_ACCUM = 8
NUM_EPOCHS = 1
LEARNING_RATE = 2e-4
WARMUP_RATIO = 0.05
LOGGING_STEPS = 50
SAVE_STEPS = 500
EVAL_STEPS = 500
SEED = 42

# LoRA config
LORA_RANK = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj"]

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("File check:")
for name, path in MODEL_DIRS.items():
    exists = os.path.isdir(path)
    print(f"  {name}: {'OK' if exists else 'MISSING'} ({path})")
print(f"  tokenizer: {'OK' if os.path.isdir(TOKENIZER_DIR) else 'MISSING'}")
print(f"  corpus: {'OK' if os.path.exists(CORPUS_PATH) else 'MISSING'}")
print(f"  eval: {'OK' if os.path.exists(EVAL_PATH) else 'MISSING'}")


## 2. Load Tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_DIR, trust_remote_code=True)

# Ensure pad token is set (needed for DataCollator)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print(f"Tokenizer vocab size: {len(tokenizer):,}")
print(f"Pad token: {tokenizer.pad_token} (id={tokenizer.pad_token_id})")
print(f"EOS token: {tokenizer.eos_token} (id={tokenizer.eos_token_id})")

## 3. Prepare Training Data

We load Armenian text, tokenize it, and pack into fixed-length sequences.
Packing concatenates tokenized lines and splits into MAX_SEQ_LEN chunks,
avoiding wasted padding tokens.

In [ ]:
def is_armenian_char(c):
    cp = ord(c)
    return (0x0530 <= cp <= 0x058F) or (0xFB13 <= cp <= 0xFB17)


def load_armenian_lines(path, max_lines, min_len=20, min_arm_ratio=0.3):
    """
    Load and filter Armenian lines from the corpus.
    Light filtering only — the corpus was already cleaned in Goal 2.
    """
    lines = []
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.strip()
            if len(line) < min_len:
                continue
            # Quick Armenian ratio check
            arm = sum(1 for c in line if is_armenian_char(c))
            alpha = sum(1 for c in line if c.isalpha())
            if alpha > 0 and arm >= 5 and (arm / alpha) >= min_arm_ratio:
                lines.append(line)
            if len(lines) >= max_lines:
                break
    return lines


print(f"Loading {MAX_TRAIN_LINES:,} lines from corpus...")
t0 = time.time()
train_lines = load_armenian_lines(CORPUS_PATH, MAX_TRAIN_LINES)
print(f"Loaded {len(train_lines):,} lines in {time.time()-t0:.0f}s")
print(f"Total characters: {sum(len(l) for l in train_lines):,}")

In [ ]:
def tokenize_and_pack(lines, tokenizer, max_seq_len, seed=42):
    """
    Tokenize all lines, concatenate into one long sequence,
    then split into fixed-length chunks.

    This is the standard 'packing' approach for causal LM training:
    - No padding waste (every token is a real token)
    - Every chunk is exactly max_seq_len tokens
    - Lines are shuffled before packing to mix topics
    """
    random.seed(seed)
    random.shuffle(lines)

    # Tokenize all lines into one flat list of token IDs
    print(f"Tokenizing {len(lines):,} lines...")
    all_ids = []
    eos_id = tokenizer.eos_token_id

    for i, line in enumerate(tqdm(lines, desc="Tokenizing")):
        ids = tokenizer.encode(line, add_special_tokens=False)
        all_ids.extend(ids)
        all_ids.append(eos_id)  # separate documents with EOS

    total_tokens = len(all_ids)
    print(f"Total tokens: {total_tokens:,}")

    # Pack into fixed-length chunks
    n_chunks = total_tokens // max_seq_len
    # Trim to exact multiple
    all_ids = all_ids[:n_chunks * max_seq_len]

    # Reshape into (n_chunks, max_seq_len)
    chunks = [all_ids[i*max_seq_len:(i+1)*max_seq_len] for i in range(n_chunks)]

    print(f"Packed into {n_chunks:,} chunks of {max_seq_len} tokens")
    print(f"Effective training tokens: {n_chunks * max_seq_len:,}")

    return chunks


train_chunks = tokenize_and_pack(train_lines, tokenizer, MAX_SEQ_LEN, SEED)

# Convert to HuggingFace Dataset
train_dataset = Dataset.from_dict({
    "input_ids": train_chunks,
    "labels": train_chunks,  # causal LM: labels = input_ids
})
train_dataset.set_format("torch")

print(f"\nTraining dataset: {len(train_dataset):,} samples")
print(f"Each sample: {MAX_SEQ_LEN} tokens")

## 4. Preparing Evaluation Data

In [ ]:
# Load eval lines (separate from training corpus)
eval_lines = load_armenian_lines(EVAL_PATH, max_lines=5000)
print(f"Eval lines: {len(eval_lines):,}")

# Pack eval data the same way (but smaller)
eval_chunks = tokenize_and_pack(eval_lines, tokenizer, MAX_SEQ_LEN, seed=123)

# Take a subset for faster eval during training
eval_subset = eval_chunks[:200]  # 200 chunks × 512 tokens = 102k tokens

eval_dataset = Dataset.from_dict({
    "input_ids": eval_subset,
    "labels": eval_subset,
})
eval_dataset.set_format("torch")
print(f"Eval dataset: {len(eval_dataset)} samples ({len(eval_dataset) * MAX_SEQ_LEN:,} tokens)")

## 5. LoRA Fine-Tuning Function

```
EMBEDDING UNFREEZING STRATEGY:
-------------------------------
By default, PEFT/LoRA freezes all base model parameters and only trains
the LoRA adapter matrices. But for tokenizer surgery recovery, the new
embeddings MUST be trained — they're only approximate initializations.

Our approach:
  1. Application of LoRA to attention layers (q/k/v/o projections)
  2. Explicitly unfreezing the input embedding layer
  3. Explicitly unfreezing the LM head (output projection)
     (For Qwen2.5 with tied embeddings, these are the same tensor)

This means we train:
  - LoRA adapters on attention (~3.5M params)
  - ALL embeddings including old ones (~163M params for 182k × 896)

Alternative considered: freezing old embeddings, only train new ones.
Rejected because:
  - HuggingFace doesn't natively support partial embedding freezing
  - The old embeddings need minor adjustments too, since the attention
    layers (via LoRA) are changing how they process token representations
  - With LoRA's small rank, the old embeddings won't change much anyway
```

In [ ]:
def setup_lora_model(model_dir, tokenizer_dir):
    """
    Loads a grafted model and applies LoRA + unfreeze embeddings.
    Returns the PEFT model ready for training.
    """
    # Loading the grafted model
    model = AutoModelForCausalLM.from_pretrained(
        model_dir,
        trust_remote_code=True,
        torch_dtype=torch.float16,
        device_map="auto")

    # Applying LoRA
    lora_config = LoraConfig(task_type=TaskType.CAUSAL_LM,
                                r=LORA_RANK,
                                lora_alpha=LORA_ALPHA,
                                lora_dropout=LORA_DROPOUT,
                                target_modules=LORA_TARGETS,
                                bias="none")

    peft_model = get_peft_model(model, lora_config)

    # Unfreezing embeddings — critical for tokenizer surgery recovery
    for name, param in peft_model.named_parameters():
        if "embed" in name.lower() or "lm_head" in name.lower():
            param.requires_grad = True
            param.data = param.data.to(torch.float32)

    # Count trainable parameters
    total_params = sum(p.numel() for p in peft_model.parameters())
    trainable_params = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)

    print(f"  Total params: {total_params:,}")
    print(f"  Trainable params: {trainable_params:,} ({trainable_params/total_params*100:.1f}%)")

    # Show what's being trained
    trained_modules = set()
    for name, param in peft_model.named_parameters():
        if param.requires_grad:
            # Get the module type (e.g., 'lora_A', 'embed_tokens')
            parts = name.split(".")
            trained_modules.add(parts[-2] if len(parts) >= 2 else parts[-1])
    print(f"  Training modules: {sorted(trained_modules)}")

    return peft_model


print("LoRA setup function defined.")

In [ ]:
def train_model(peft_model, train_dataset, eval_dataset, output_name):
    """
    Runs LoRA fine-tuning using HuggingFace Trainer.
    """
    output_path = os.path.join(OUTPUT_DIR, output_name)

    training_args = TrainingArguments(output_dir=output_path,
                                      num_train_epochs=NUM_EPOCHS,
                                      per_device_train_batch_size=BATCH_SIZE,
                                      per_device_eval_batch_size=BATCH_SIZE,
                                      gradient_accumulation_steps=GRAD_ACCUM,
                                      learning_rate=LEARNING_RATE,
                                      lr_scheduler_type="cosine",
                                      warmup_ratio=WARMUP_RATIO,
                                      weight_decay=0.01,
                                      fp16=True,
                                      logging_steps=LOGGING_STEPS,
                                      save_steps=SAVE_STEPS,
                                      eval_strategy="steps",
                                      eval_steps=EVAL_STEPS,
                                      save_total_limit=2,
                                      load_best_model_at_end=True,
                                      metric_for_best_model="eval_loss",
                                      greater_is_better=False,
                                      seed=SEED,
                                      report_to="none",  # "wandb" if logging is desired
                                      dataloader_num_workers=2,
                                      remove_unused_columns=False)

    # Data collator for causal LM (labels = input_ids, shifted internally)
    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)  # causal LM, not masked LM


    trainer = Trainer(
        model=peft_model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        data_collator=data_collator)

    print(f"\nStarting training: {output_name}")
    print(f"  Steps: {len(train_dataset) // (BATCH_SIZE * GRAD_ACCUM):,}")
    print(f"  Effective batch size: {BATCH_SIZE * GRAD_ACCUM}")

    t0 = time.time()
    train_result = trainer.train()
    elapsed = time.time() - t0

    print(f"\nTraining completed in {elapsed/60:.1f} minutes")
    print(f"  Final train loss: {train_result.training_loss:.4f}")

    # Save the final model
    trainer.save_model(output_path)
    tokenizer.save_pretrained(output_path)
    print(f"  Saved to: {output_path}")

    # Get eval metrics
    eval_results = trainer.evaluate()
    print(f"  Final eval loss: {eval_results['eval_loss']:.4f}")
    print(f"  Final eval PPL: {math.exp(eval_results['eval_loss']):.2f}")

    return {"name": output_name,
        "train_loss": train_result.training_loss,
        "eval_loss": eval_results["eval_loss"],
        "eval_ppl": math.exp(eval_results["eval_loss"]),
        "train_time_min": elapsed / 60,
        "train_steps": train_result.global_step,
        "log_history": trainer.state.log_history}

## 6. Training All 3 Init Strategies

We fine-tune each grafted model variant under identical conditions
to fairly compare which initialization strategy recovers fastest.

In [ ]:
# Train heuristic-init first (expected best from Goal 3)
all_results = {}

for strategy_name in ["heuristic_init", "mean_init", "nearest_init"]:
    model_dir = MODEL_DIRS[strategy_name]
    if not os.path.isdir(model_dir):
        print(f"\nSKIPPING {strategy_name} — directory not found: {model_dir}")
        continue

    print(f"\n{'='*70}")
    print(f"TRAINING: {strategy_name}")
    print(f"{'='*70}")

    # Setup
    print(f"Loading model from {model_dir}...")
    peft_model = setup_lora_model(model_dir, TOKENIZER_DIR)

    # Train
    result = train_model(peft_model, train_dataset, eval_dataset,
        output_name=f"lora_{strategy_name}")
    all_results[strategy_name] = result

    # Freeing up GPU memory before next model run
    del peft_model
    torch.cuda.empty_cache()

    print(f"\n{strategy_name} complete: eval_ppl={result['eval_ppl']:.2f}")

## 7. Results Comparison

In [ ]:
# Goal 3 pre-fine-tuning perplexities for reference
PRE_FT_PPL = {
    "mean_init": 216406.1, "heuristic_init": 24484.3, "nearest_init": 96967.5}

print("=" * 70)
print("RECOVERY FINE-TUNING RESULTS")
print("=" * 70)
print(f"\n{'Strategy':<20} {'Pre-FT PPL':>12} {'Post-FT PPL':>12} {'Reduction':>10} {'Time (min)':>10}")
print("-" * 66)

for name in ["heuristic_init", "mean_init", "nearest_init"]:
    if name not in all_results:
        continue
    r = all_results[name]
    pre_ppl = PRE_FT_PPL.get(name, float("inf"))
    post_ppl = r["eval_ppl"]
    reduction = (1 - post_ppl / pre_ppl) * 100 if pre_ppl > 0 else 0
    print(f"{name:<20} {pre_ppl:>12,.1f} {post_ppl:>12.2f} {reduction:>9.1f}% {r['train_time_min']:>10.1f}")

# determining the best
if all_results:
    best_name = min(all_results, key=lambda k: all_results[k]["eval_ppl"])
    best_ppl = all_results[best_name]["eval_ppl"]
    print(f"\nBest model: {best_name} with eval perplexity {best_ppl:.2f}")

In [ ]:
# Plot training loss curves for all strategies
try:
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    colors = {"heuristic_init": "#1D9E75", "mean_init": "#3266AD", "nearest_init": "#D85A30"}

    for name, result in all_results.items():
        # Extract training loss from log history
        train_logs = [(l["step"], l["loss"]) for l in result["log_history"] if "loss" in l and "eval" not in l]
        eval_logs = [(l["step"], l["eval_loss"]) for l in result["log_history"] if "eval_loss" in l]

        if train_logs:
            steps, losses = zip(*train_logs)
            axes[0].plot(steps, losses, label=name, color=colors.get(name, "gray"), linewidth=1.5)

        if eval_logs:
            steps, losses = zip(*eval_logs)
            axes[1].plot(steps, losses, label=name, color=colors.get(name, "gray"), linewidth=1.5, marker="o", markersize=4)

    axes[0].set_xlabel("Step")
    axes[0].set_ylabel("Training Loss")
    axes[0].set_title("Training Loss")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].set_xlabel("Step")
    axes[1].set_ylabel("Eval Loss")
    axes[1].set_title("Evaluation Loss")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "training_curves.png"), dpi=150, bbox_inches="tight")
    plt.show()
    print("Chart saved.")
except ImportError:
    print("matplotlib not available — skipping chart")

## 8. Generation Test

Quick qualitative check: can the fine-tuned model generate
coherent Armenian text?

In [ ]:
def generate_sample(model_path, tokenizer, prompts, max_new_tokens=100):
    """
    Load a fine-tuned model and generate text from Armenian prompts.
    """
    from peft import PeftModel

    # Load base model + LoRA adapter
    base_model = AutoModelForCausalLM.from_pretrained(
        model_path,
        trust_remote_code=True,
        torch_dtype=torch.float16,
        device_map="auto",
    )

    base_model.eval()

    print(f"Generating from: {model_path}\n")
    for prompt in prompts:
        inputs = tokenizer(prompt, return_tensors="pt").to(base_model.device)
        with torch.no_grad():
            outputs = base_model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=0.7,
                top_p=0.9,
                repetition_penalty=1.2,
            )
        generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
        print(f"Prompt: {prompt}")
        print(f"Generated: {generated}")
        print("-" * 60)

    del base_model
    torch.cuda.empty_cache()


# Test prompts (short Armenian phrases to continue)
# These are common openings — the model should produce grammatical continuations
test_prompts = ["Հայաստանը գտնվում է", "Հայաստանի մայրաքաղաք", "2017 թվականին սկսվել է"]

# Generatimg from the best model
if all_results:
    best_name = min(all_results, key=lambda k: all_results[k]["eval_ppl"])
    best_path = os.path.join(OUTPUT_DIR, f"lora_{best_name}")
    generate_sample(best_path, tokenizer, test_prompts)

## 9. Save Final Results

In [ ]:
goal4_results = {"training_config": {"corpus": CORPUS_PATH,
                  "train_lines": MAX_TRAIN_LINES,
                  "max_seq_len": MAX_SEQ_LEN,
                  "batch_size": BATCH_SIZE,
                  "grad_accum": GRAD_ACCUM,
                  "effective_batch": BATCH_SIZE * GRAD_ACCUM,
                  "epochs": NUM_EPOCHS,
                  "learning_rate": LEARNING_RATE,
                  "lora_rank": LORA_RANK,
                  "lora_alpha": LORA_ALPHA,
                  "lora_targets": LORA_TARGETS},
                    "pre_finetuning_ppl": PRE_FT_PPL,
                    "results": {}}

for name, r in all_results.items():
    goal4_results["results"][name] = {"train_loss": r["train_loss"],
                                      "eval_loss": r["eval_loss"],
                                      "eval_ppl": r["eval_ppl"],
                                      "train_time_min": r["train_time_min"],
                                      "train_steps": r["train_steps"]}

results_path = os.path.join(OUTPUT_DIR, "goal4_results.json")
with open(results_path, "w") as f:
    json.dump(goal4_results, f, indent=2)

print(f"Results saved to: {results_path}")
print(f"\nFine-tuned model directories:")
for name in all_results:
    path = os.path.join(OUTPUT_DIR, f"lora_{name}")
    print(f"  {name}: {path}/")

print(f"\n--- Goal 4 complete! Proceed to Goal 5 (evaluation). ---")

---

## Summary of Design Decisions

| Decision | Choice | Rationale |
|----------|--------|-----------|
| Fine-tuning method | LoRA + unfrozen embeddings | LoRA preserves English; embeddings must train |
| LoRA rank | 16 | Good balance for 0.5B model; standard in literature |
| LoRA alpha | 32 (2×rank) | Standard scaling factor |
| LoRA targets | q,k,v,o projections | Adapts attention to process Armenian tokens |
| Training data | 500k lines (~34M tokens) | Sufficient for recovery; fits time budget |
| Sequence length | 512 | Covers ~300 Armenian words; fits T4 memory |
| Batch size | 4 × 8 accum = 32 effective | Standard for LoRA fine-tuning |
| Learning rate | 2e-4 cosine | Standard LoRA LR (QLoRA paper) |
| Epochs | 1 | Recovery doesn't need multiple passes |
| Data packing | Concatenate + split into chunks | Eliminates padding waste |
| Precision | fp16 | Halves memory; standard for T4/A100 |